## 📂 P.3: Data Structures (Nested Extraction)
### Conceptual Lecture: Structural Indexing vs Key Queries
In Python:
- **Lists (`list`)** are ordered sequences of objects accessed by **integer indices** starting at `0`.
- **Dictionaries (`dict`)** are unordered mappings of key-value pairs accessed by **keys** (typically strings or integers).

When navigating deep JSON data structures, you must identify the data type of the active level. If the level is enclosed in brackets `[...]`, it is a **List**; you must query it using an index like `[0]`. If it is enclosed in curly braces `{...}`, it is a **Dictionary**; you must query it using a key like `["key"]`.

### 🧪 Discovery Lab: The Index vs Key TypeError
We have a typical response from an LLM API. We want to extract the content of the assistant's message.

In [ ]:
payload = {
    "model": "qwen",
    "messages": [
        {"role": "user", "content": "hello"},
        {"role": "assistant", "content": "hi"}
    ]
}

try:
    # Try to extract the assistant content blindly using key query directly
    content = payload["messages"]["content"]
except TypeError as e:
    print(f"CRASHED as expected: {e}")

#### The Algorithmic Necessity
`messages` is a **list** of dictionaries, not a dictionary. You cannot query a list directly using a string key. You must first index the list numerically (`[1]` for the second item) and then extract the key from that dictionary.

#### The Rosetta Stone
```python
# Step 1: Access the list
msg_list = payload["messages"]  # msg_list is a list
# Step 2: Index the 2nd message (assistant)
assistant_msg = msg_list[1]  # assistant_msg is a dict
# Step 3: Extract the content string
content = assistant_msg["content"]  # content is "hi"

# One-line consolidated query:
content = payload["messages"][1]["content"]
```

In [ ]:
# TODO: Correctly extract "hi" from the payload using combined list index and dictionary key access.
assistant_content = ""

# Your extraction here:


assert assistant_content == "hi", f"Expected 'hi', got {assistant_content}"
print("Success! Nested dictionary extraction mastered.")

### 🚀 Active Lab Exercise: Harvesting Deep Tool Calls
Modern LLM API payloads are heavily nested, especially when returning tool execution intents.

**Your Task:** Extract the name of the tool and the city parameter from the nested dictionary payload below.
The targeted fields are:
1. Tool Name: Should be `"get_weather"`
2. Location Argument: Should be `"London"` (Note: The `arguments` value is a serialized JSON string! You must extract it and parse it using `json.loads` to get the key).

In [ ]:
import json

response = {
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": None,
                "tool_calls": [
                    {
                        "id": "call_abc123",
                        "type": "function",
                        "function": {
                            "name": "get_weather",
                            "arguments": '{"location": "London"}'
                        }
                    }
                ]
            }
        }
    ]
}

# TODO: Extract the function name and arguments string, parse it, and extract the location.
tool_name = ""
location = ""

# Your extraction and parsing code here:


assert tool_name == "get_weather", f"Expected 'get_weather', got {tool_name}"
assert location == "London", f"Expected 'London', got {location}"
print(f"Success! Safely extracted tool call: {tool_name} for {location}.")

---
### 🚀 Active Lab Exercise 3: Dictionary Comprehensions (Lookups)
To optimize performance, engineers never loop over a list to find an item by ID (an $O(N)$ lookup bottleneck). Instead, they index lists into lookup dictionaries ($O(1)$ constant time lookups).

**Your Task:** Transform a list of nested tool-call dictionary definitions into a lookup registry using a single **Dictionary Comprehension**.

In [ ]:
tools = [
    {"id": "call_01", "name": "web_search", "params": ["query"]},
    {"id": "call_02", "name": "calculator", "params": ["expression"]},
    {"id": "call_03", "name": "retriever", "params": ["doc_id"]}
]

# TODO: Build a lookup dictionary 'registry' mapping tool 'id' -> full tool dictionary.
registry = {}

# Your dictionary comprehension here:


assert len(registry) == 3
assert registry["call_02"]["name"] == "calculator"
assert "call_04" not in registry
print("Success! Dictionary lookup registry compiled in O(1) space.")

---
### 🚀 Active Lab Exercise 4: Schema Validation Contract
Before dispatching execution commands to a tool, we must validate the payload contract. If a key is missing, or contains invalid data types, the system must detect it without crashing.

**Your Task:** Implement a strict validation checker `validate_tool_schema(payload)`.

In [ ]:
def validate_tool_schema(payload) -> bool:
    # TODO: Verify the dictionary conforms to these criteria:
    # 1. payload must be an actual dict.
    # 2. "id" and "name" keys must exist and be strings.
    # 3. "params" key must exist, be a list, and all elements inside "params" must be strings.
    # Return True if valid, False otherwise. Prevent errors from raising using safe checks!
    pass

# Verification:
assert validate_tool_schema({"id": "1", "name": "test", "params": ["a"]}) == True
assert validate_tool_schema({"id": "1", "name": "test"}) == False # Missing params
assert validate_tool_schema({"id": 1, "name": "test", "params": []}) == False # id not string
assert validate_tool_schema("not a dict") == False
assert validate_tool_schema({"id": "1", "name": "test", "params": [1, 2]}) == False # params contains non-strings
print("Success! API validation schema contract is secure.")